In [1]:
import cdsapi

dataset = "reanalysis-era5-single-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": [
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "sea_surface_temperature"
    ],
    "year": ["1976"],
    "month": ["11"],
    "day": ["11"],
    "time": [
        "00:00", "01:00", "02:00",
        "03:00", "04:00", "05:00",
        "06:00", "07:00", "08:00",
        "09:00", "10:00", "11:00",
        "12:00", "13:00", "14:00",
        "15:00", "16:00", "17:00",
        "18:00", "19:00", "20:00",
        "21:00", "22:00", "23:00"
    ],
    "data_format": "netcdf",
    "download_format": "unarchived"
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()


2026-03-18 12:06:31,886 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-03-18 12:06:31,887 INFO Request ID is 9252de6f-e86b-4337-8893-cf8360f4f360
2026-03-18 12:06:32,076 INFO status has been updated to accepted
2026-03-18 12:06:46,595 INFO status has been updated to running
2026-03-18 12:07:23,263 INFO status has been updated to successful


'b4af554fcb1b370f47c046b04b935903.nc'

In [ ]:
# -------------------------------------------------------
# Load data, compute wind speed & direction, convert SST to °C
# -------------------------------------------------------
import xarray as xr
import numpy as np

ds = xr.open_dataset('era5_daily_data/ERA5_daily_334N158W_1990_2020.nc')

# Select nearest grid point
ds_pt = ds.sel(latitude=lat_pt, longitude=lon_pt, method='nearest')

# SST: Kelvin -> Celsius
sst = ds_pt['sst'] - 273.15
sst.attrs['units'] = '°C'

# Wind speed and direction from u, v components
u10 = ds_pt['u10']
v10 = ds_pt['v10']
wspd = np.sqrt(u10**2 + v10**2)
wdir = np.degrees(np.arctan2(-u10, -v10)) % 360  # meteorological convention
wspd.attrs['units'] = 'm/s'
wdir.attrs['units'] = 'degrees'

print('SST:       ', sst)
print('Wind speed:', wspd)
print('Wind dir:  ', wdir)


In [ ]:
# -------------------------------------------------------
# Download ERA5 daily snapshot (00:00 UTC), year-by-year 1990-2020
# Variables: SST, winds, and ocean waves
# 31 requests with skip-if-exists so restartable
# -------------------------------------------------------
import cdsapi
import os

lon_pt = -158.0
lat_pt =  33.4
dbox   =  0.1

os.makedirs('era5_daily_data', exist_ok=True)
c = cdsapi.Client()

for year in range(1990, 2021):
    yr      = str(year)
    outfile = f'era5_daily_data/ERA5_snap_33N158W_{yr}.nc'
    if os.path.exists(outfile):
        print(f'{yr}: already exists, skipping')
        continue
    print(f'Downloading {yr}...')
    c.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable': [
                'sea_surface_temperature',
                '10m_u_component_of_wind',
                '10m_v_component_of_wind',
                'significant_height_of_combined_wind_waves_and_swell',
                'mean_wave_direction',
                'mean_wave_period',
                'peak_wave_period',
            ],
            'year':  yr,
            'month': [f'{m:02d}' for m in range(1, 13)],
            'day':   [f'{d:02d}' for d in range(1, 32)],
            'time':  '00:00',
            'area':  [lat_pt + dbox, lon_pt - dbox, lat_pt - dbox, lon_pt + dbox],
            'data_format': 'netcdf',
            'download_format': 'unarchived',
        },
        outfile
    )
    print(f'  -> saved {outfile}')

print('All downloads complete.')

In [ ]:
# -------------------------------------------------------
# Load data, compute wind speed & direction, convert SST to °C
# -------------------------------------------------------
import xarray as xr
import numpy as np

ds = xr.open_dataset('era5_daily_data/ERA5_daily_334N158W_1990_2020.nc')

# Select nearest grid point
ds_pt = ds.sel(latitude=lat_pt, longitude=lon_pt, method='nearest')

# SST: Kelvin -> Celsius
sst = ds_pt['sst'] - 273.15
sst.attrs['units'] = '°C'

# Wind speed and direction from u, v components
u10 = ds_pt['u10']
v10 = ds_pt['v10']
wspd = np.sqrt(u10**2 + v10**2)
wdir = np.degrees(np.arctan2(-u10, -v10)) % 360  # meteorological convention
wspd.attrs['units'] = 'm/s'
wdir.attrs['units'] = 'degrees'

print('SST:       ', sst)
print('Wind speed:', wspd)
print('Wind dir:  ', wdir)


In [ ]:
# -------------------------------------------------------
# Load 30-year snapshot data and plot SST + wind speed
# -------------------------------------------------------
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

# --- Load files one at a time, build pandas Series ---
files = sorted(glob.glob('era5_daily_data/ERA5_snap_*158W_*.nc'))
print(f'Found {len(files)} files')

sst_list, u_list, v_list, time_list = [], [], [], []

for f in files:
    ds = xr.open_dataset(f, drop_variables=['number', 'expver'])
    ds = ds.squeeze()
    time_list.append(ds['valid_time'].values)
    sst_list.append(ds['sst'].values)
    u_list.append(ds['u10'].values)
    v_list.append(ds['v10'].values)
    ds.close()

# Concatenate into 1-D numpy arrays
time = np.concatenate(time_list)
sst  = np.concatenate(sst_list) - 273.15   # K -> °C
u10  = np.concatenate(u_list)
v10  = np.concatenate(v_list)
wspd = np.sqrt(u10**2 + v10**2)

print(f'Total time steps: {len(time)}')

# Build pandas Series for rolling mean
idx       = pd.DatetimeIndex(time)
sst_s     = pd.Series(sst,  index=idx)
wspd_s    = pd.Series(wspd, index=idx)
sst_roll  = sst_s.rolling(30, center=True, min_periods=15).mean()
wspd_roll = wspd_s.rolling(30, center=True, min_periods=15).mean()

# --- Plot ---
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.suptitle('ERA5 at 33.4°N, 158°W  |  1990–2020  (00:00 UTC daily snapshot)',
             fontsize=13, fontweight='bold')

# SST
ax = axes[0]
ax.plot(idx, sst_s,   color='#aec7e8', lw=0.6, alpha=0.7, label='Daily')
ax.plot(idx, sst_roll, color='#1f77b4', lw=1.8, label='30-day rolling average')
ax.set_ylabel('SST (°C)', fontsize=11)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', fontsize=9)

# Wind speed
ax = axes[1]
ax.plot(idx, wspd_s,   color='#c7e9c0', lw=0.6, alpha=0.7, label='Daily')
ax.plot(idx, wspd_roll, color='#2ca02c', lw=1.8, label='30-day rolling average')
ax.set_ylabel('Wind speed (m/s)', fontsize=11)
ax.set_ylim(0, 20)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', fontsize=9)
ax.set_xlabel('Year', fontsize=11)

plt.tight_layout()
os.makedirs('figs', exist_ok=True)
plt.savefig('figs/ERA5_SST_wind_30yr.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figs/ERA5_SST_wind_30yr.png')


In [ ]:
# -------------------------------------------------------
# Download ERA5 hourly — all air/sea interaction fields
# Location: 33.4N, 158W | Period: Nov 1 2025 – Mar 14 2026
# Split into 2 requests (by year) to stay within CDS field limits
#
# NOTE on flux variables (latent, sensible, radiation, stress,
# precip, evaporation): ERA5 stores these as hourly ACCUMULATIONS
# (J/m² or metres). To convert to W/m² divide by 3600.
# -------------------------------------------------------
import cdsapi
import os

lon_pt = -158.0
lat_pt =  33.4
dbox   =  0.1

ALL_VARS = [
    # --- Winds ---
    '10m_u_component_of_wind',
    '10m_v_component_of_wind',
    'instantaneous_10m_wind_gust',
    # --- Temperature / humidity ---
    'sea_surface_temperature',
    'skin_temperature',
    '2m_temperature',
    '2m_dewpoint_temperature',
    # --- Heat fluxes (accumulated J/m²; divide by 3600 -> W/m²) ---
    'mean_surface_latent_heat_flux',
    'mean_surface_sensible_heat_flux',
    'mean_surface_net_short_wave_radiation_flux',
    'mean_surface_net_long_wave_radiation_flux',
    'mean_surface_downward_short_wave_radiation_flux',
    'mean_surface_downward_long_wave_radiation_flux',
    # --- Wind stress (accumulated N/m²·s; divide by 3600 -> N/m²) ---
    'mean_eastward_turbulent_surface_stress',
    'mean_northward_turbulent_surface_stress',
    # --- Precip / evaporation (accumulated metres) ---
    'total_precipitation',
    'evaporation',
    # --- Boundary layer ---
    'boundary_layer_height',
    'total_column_water_vapour',
    # --- Waves ---
    'significant_height_of_combined_wind_waves_and_swell',
    'mean_wave_direction',
    'mean_wave_period',
    'peak_wave_period',
]

# Two chunks: 2025 (Nov-Dec) and 2026 (Jan-Mar)
CHUNKS = [
    {
        'label':  '2025_NovDec',
        'year':   '2025',
        'months': ['11', '12'],
        'days':   [f'{d:02d}' for d in range(1, 32)],
    },
    {
        'label':  '2026_JanMar',
        'year':   '2026',
        'months': ['01', '02', '03'],
        'days':   [f'{d:02d}' for d in range(1, 32)],
    },
]

os.makedirs('era5_airsea_data', exist_ok=True)
c = cdsapi.Client()

for chunk in CHUNKS:
    outfile = f"era5_airsea_data/ERA5_airsea_334N158W_{chunk['label']}.nc"
    if os.path.exists(outfile):
        print(f"{chunk['label']}: already exists, skipping")
        continue
    print(f"Downloading {chunk['label']}...")
    c.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable':     ALL_VARS,
            'year':         chunk['year'],
            'month':        chunk['months'],
            'day':          chunk['days'],
            'time':         [f'{h:02d}:00' for h in range(24)],
            'area':         [lat_pt + dbox, lon_pt - dbox,
                             lat_pt - dbox, lon_pt + dbox],
            'data_format':      'netcdf',
            'download_format':  'unarchived',
        },
        outfile
    )
    print(f'  -> saved {outfile}')

print('Done.')
